# PyTorch's Hidden Toolkit — Utilities You Should Know

**What you'll learn:**
- Weight parametrizations (symmetric, positive, orthogonal)
- Model pruning strategies and workflows
- Sequence packing for efficient RNN processing
- NestedTensors for variable-length data
- Conv-BN fusion, skip_init, and parameter flattening

**Prerequisites:** Familiar with nn.Module, training loops, and basic architectures

In [ ]:
import torch
import torch.nn as nn
import torch.nn.utils.parametrize as parametrize
import torch.nn.utils.prune as prune
print(f"PyTorch version: {torch.__version__}")

## Part 1: Weight Parametrizations

Parametrizations let you apply constraints to module parameters transparently. Instead of manually enforcing constraints (e.g., "this weight must be symmetric"), you register a parametrization and PyTorch handles it automatically during forward passes.

In [ ]:
# Custom parametrization: make a weight matrix symmetric
class Symmetric(nn.Module):
    def forward(self, W):
        return (W + W.T) / 2

layer = nn.Linear(5, 5)
print("Before parametrization:")
print(f"  W is symmetric: {torch.allclose(layer.weight, layer.weight.T)}")

parametrize.register_parametrization(layer, "weight", Symmetric())
print("\nAfter parametrization:")
print(f"  W is symmetric: {torch.allclose(layer.weight, layer.weight.T)}")
print(f"  Original weight stored in: layer.parametrizations.weight.original")
print(f"  Effective weight:\n{layer.weight}")

### Positive Definite Parametrization

For covariance matrices or other cases where you need a matrix to be positive definite:

In [ ]:
class PositiveDefinite(nn.Module):
    def forward(self, W):
        # W @ W^T is always positive semi-definite; add small diagonal for strict PD
        return W @ W.T + 0.01 * torch.eye(W.shape[0])

layer_pd = nn.Linear(4, 4, bias=False)
parametrize.register_parametrization(layer_pd, "weight", PositiveDefinite())

W = layer_pd.weight
eigenvalues = torch.linalg.eigvalsh(W)
print(f"Effective weight:\n{W}")
print(f"\nEigenvalues: {eigenvalues}")
print(f"All positive: {(eigenvalues > 0).all()}")

## Built-in Parametrizations

PyTorch ships several common parametrizations out of the box:

In [ ]:
from torch.nn.utils.parametrizations import spectral_norm, orthogonal

# Spectral normalization: constrains the spectral norm (largest singular value) to 1
layer_sn = nn.Linear(8, 8)
layer_sn = spectral_norm(layer_sn)
sigma = torch.linalg.svdvals(layer_sn.weight)[0]
print(f"Spectral norm after parametrization: {sigma:.4f}")

# Orthogonal parametrization: makes weight an orthogonal matrix
layer_orth = nn.Linear(5, 5)
layer_orth = orthogonal(layer_orth)
W = layer_orth.weight
identity_check = W @ W.T
print(f"\nOrthogonal W @ W^T (should be ~identity):")
print(identity_check.data.round(decimals=3))

## Part 2: Model Pruning

Pruning removes parameters (sets them to zero) to make models smaller and faster. PyTorch provides structured and unstructured pruning methods.

In [ ]:
# L1 unstructured pruning: zero out smallest weights individually
linear = nn.Linear(10, 8)
print("Before pruning:")
print(f"  Non-zero weights: {(linear.weight != 0).sum().item()}/{linear.weight.numel()}")

# Prune 30% of weights with smallest L1 magnitude
prune.l1_unstructured(linear, name="weight", amount=0.3)

print("\nAfter 30% L1 unstructured pruning:")
print(f"  Non-zero weights: {(linear.weight != 0).sum().item()}/{linear.weight.numel()}")
print(f"  Sparsity: {100 * (linear.weight == 0).sum().item() / linear.weight.numel():.0f}%")
print(f"\nPruning creates a mask:")
print(f"  linear.weight_mask shape: {linear.weight_mask.shape}")
print(f"  Mask zeros: {(linear.weight_mask == 0).sum().item()}")

### Structured Pruning

Unstructured pruning zeros individual weights. **Structured pruning** removes entire neurons, channels, or filters — which actually speeds up inference (sparse individual weights don't help much without sparse hardware).

In [ ]:
# Structured pruning: remove entire output neurons
linear2 = nn.Linear(10, 8)
print(f"Before: weight shape = {linear2.weight.shape}")

# Remove 3 output neurons with smallest L2 norm
prune.ln_structured(linear2, name="weight", amount=3, n=2, dim=0)

print(f"After structured pruning (3 of 8 output neurons):")
print(f"  Weight shape: {linear2.weight.shape} (shape unchanged)")
print(f"  Zero rows (pruned neurons):")
for i in range(8):
    is_zero = (linear2.weight[i] == 0).all().item()
    if is_zero:
        print(f"    Neuron {i}: PRUNED")

### Global Pruning

Instead of pruning each layer independently, **global pruning** finds the globally least important weights across all layers. This is usually better because it allows important layers to keep more weights.

In [ ]:
# Global pruning across a full model
class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 4)

net = SmallNet()

# Define which parameters to prune globally
parameters_to_prune = (
    (net.fc1, 'weight'),
    (net.fc2, 'weight'),
    (net.fc3, 'weight'),
)

# Prune 50% of all weights globally
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.5,
)

# Check sparsity per layer — global pruning distributes unevenly!
for name, module in net.named_modules():
    if isinstance(module, nn.Linear):
        total = module.weight.numel()
        zeros = (module.weight == 0).sum().item()
        print(f"{name}: {zeros}/{total} pruned ({100*zeros/total:.0f}%)")

total_params = sum(m.weight.numel() for m in [net.fc1, net.fc2, net.fc3])
total_zeros = sum((m.weight == 0).sum().item() for m in [net.fc1, net.fc2, net.fc3])
print(f"\nGlobal: {total_zeros}/{total_params} pruned ({100*total_zeros/total_params:.0f}%)")

### The Pruning Workflow: Train → Prune → Fine-tune

Pruning works best as a post-training step. The typical workflow is:
1. **Train** the model to convergence
2. **Prune** to remove less important weights
3. **Fine-tune** the pruned model to recover accuracy
4. **Make permanent** by removing the pruning reparametrization

In [ ]:
# Complete pruning workflow
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 2)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Step 1: "Train" (simulate with random data)
torch.manual_seed(42)
model = TinyClassifier()
X = torch.randn(200, 20)
Y = torch.randint(0, 2, (200,))

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(50):
    loss = nn.functional.cross_entropy(model(X), Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
accuracy_before = (model(X).argmax(1) == Y).float().mean()
print(f"Before pruning — Accuracy: {accuracy_before:.2%}")

# Step 2: Prune 40% globally
prune.global_unstructured(
    [(model.fc1, 'weight'), (model.fc2, 'weight'), (model.fc3, 'weight')],
    pruning_method=prune.L1Unstructured,
    amount=0.4,
)
accuracy_after_prune = (model(X).argmax(1) == Y).float().mean()
print(f"After pruning  — Accuracy: {accuracy_after_prune:.2%}")

# Step 3: Fine-tune
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(30):
    loss = nn.functional.cross_entropy(model(X), Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
accuracy_after_finetune = (model(X).argmax(1) == Y).float().mean()
print(f"After fine-tune — Accuracy: {accuracy_after_finetune:.2%}")

# Step 4: Make pruning permanent
for module in [model.fc1, model.fc2, model.fc3]:
    prune.remove(module, 'weight')
print(f"\nPruning masks removed — weights are now permanently sparse")

## Part 3: Sequence Packing for RNNs

When processing sequences of different lengths, you can either pad (wasteful) or **pack** (efficient). Packing removes the padding and tells the RNN to process only real tokens.

In [ ]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

# Simulate variable-length sequences (e.g., sentences of different lengths)
sequences = [
    torch.randn(8, 16),   # length 8
    torch.randn(5, 16),   # length 5
    torch.randn(12, 16),  # length 12
    torch.randn(3, 16),   # length 3
]
lengths = torch.tensor([8, 5, 12, 3])

# Step 1: Pad to same length
padded = pad_sequence(sequences, batch_first=True)
print(f"Padded shape: {padded.shape}")
print(f"  (batch=4, max_len=12, features=16)")
print(f"  Wasted computation: {(12-8) + (12-5) + (12-3)} padding positions")

# Step 2: Sort by length (required for packing)
sorted_lengths, sorted_idx = lengths.sort(descending=True)
sorted_padded = padded[sorted_idx]

# Step 3: Pack — removes padding
packed = pack_padded_sequence(sorted_padded, sorted_lengths, batch_first=True)
print(f"\nPacked data shape: {packed.data.shape}")
print(f"  Only {packed.data.shape[0]} actual positions (vs {4*12}=48 padded)")

# Step 4: Run through LSTM
lstm = nn.LSTM(input_size=16, hidden_size=32, batch_first=True)
packed_out, (h_n, c_n) = lstm(packed)

# Step 5: Unpack back to padded
output, output_lengths = pad_packed_sequence(packed_out, batch_first=True)
print(f"\nLSTM output shape: {output.shape}")
print(f"Output lengths: {output_lengths}")

## Part 4: NestedTensors

NestedTensors are PyTorch's native way to handle variable-length data **without padding**. Each element can have a different size along certain dimensions.

In [ ]:
# NestedTensors: variable-length sequences without padding
seq1 = torch.randn(3, 8)   # 3 tokens, 8 features
seq2 = torch.randn(5, 8)   # 5 tokens
seq3 = torch.randn(2, 8)   # 2 tokens

nt = torch.nested.nested_tensor([seq1, seq2, seq3], layout=torch.jagged)
print(f"NestedTensor: {nt}")
print(f"  Outer shape: batch=3")
print(f"  Inner shapes: [{seq1.shape}, {seq2.shape}, {seq3.shape}]")

# Linear layers work on nested tensors
linear = nn.Linear(8, 16)
out_nt = linear(nt)
print(f"\nAfter linear(8→16):")
print(f"  Output is still a NestedTensor")

## Part 5: Conv-BN Fusion for Inference

During inference, a Conv2d followed by BatchNorm2d can be **fused into a single Conv2d** — mathematically equivalent but faster (one kernel instead of two).

**The math:** BatchNorm applies `y = (x - mean) / sqrt(var + eps) * gamma + beta`, which is a linear transform that can be absorbed into the convolution weights and bias.

In [ ]:
# Conv-BN fusion demonstration
from torch.nn.utils import fuse_conv_bn_eval

# Create a conv + batchnorm pair
conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
bn = nn.BatchNorm2d(16)

# Train the batchnorm to get meaningful running stats
bn.train()
for _ in range(20):
    fake_input = torch.randn(8, 3, 32, 32)
    bn(conv(fake_input))

# Switch to eval mode (required for fusion)
conv.eval()
bn.eval()

# Fuse!
fused_conv = fuse_conv_bn_eval(conv, bn)

# Verify they produce the same output
test_input = torch.randn(4, 3, 32, 32)
with torch.no_grad():
    original_out = bn(conv(test_input))
    fused_out = fused_conv(test_input)

print(f"Original: Conv2d + BatchNorm2d (2 operations)")
print(f"Fused:    Conv2d (1 operation)")
print(f"Outputs match: {torch.allclose(original_out, fused_out, atol=1e-5)}")
print(f"Max difference: {(original_out - fused_out).abs().max():.2e}")

## Part 6: skip_init for Fast Model Creation

When you're going to load pre-trained weights anyway, initializing random weights is wasted work. `skip_init` creates modules without running the default initialization.

In [ ]:
import torch.utils.benchmark as bench_util
import time

# Compare normal init vs skip_init
n = 4096

# Normal initialization
start = time.perf_counter()
normal_linear = nn.Linear(n, n)
normal_time = time.perf_counter() - start

# skip_init
start = time.perf_counter()
skip_linear = nn.utils.skip_init(nn.Linear, n, n)
skip_time = time.perf_counter() - start

print(f"nn.Linear({n}, {n}):")
print(f"  Normal init: {normal_time*1000:.2f} ms")
print(f"  skip_init:   {skip_time*1000:.2f} ms")
print(f"  Speedup:     {normal_time/skip_time:.1f}x")
print(f"\nNote: skip_init is most useful for large models where")
print(f"you're immediately loading pre-trained weights.")

## Part 7: parameters_to_vector and vector_to_parameters

Flatten all model parameters into a single vector — useful for optimization algorithms, model comparison, and regularization.

In [ ]:
from torch.nn.utils import parameters_to_vector, vector_to_parameters

model_a = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5))
model_b = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5))

# Flatten all parameters to a single vector
vec_a = parameters_to_vector(model_a.parameters())
vec_b = parameters_to_vector(model_b.parameters())

print(f"Model A total parameters: {vec_a.numel()}")
print(f"Model B total parameters: {vec_b.numel()}")

# Compute distance between two models
distance = (vec_a - vec_b).norm()
print(f"\nL2 distance between models: {distance:.4f}")

# Interpolate between models (useful for model averaging)
alpha = 0.5
interpolated_vec = alpha * vec_a + (1 - alpha) * vec_b

# Load interpolated parameters back into a model
model_c = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5))
vector_to_parameters(interpolated_vec, model_c.parameters())

vec_c = parameters_to_vector(model_c.parameters())
print(f"\nInterpolated model is midpoint: {torch.allclose(vec_c, (vec_a + vec_b) / 2)}")

## 🏋️ Try It Yourself: Prune and Measure

**Exercise:** Create a 3-layer MLP, train it on synthetic data, then:
1. Measure accuracy before pruning
2. Apply 50% global L1 pruning
3. Measure accuracy after pruning (expect some drop)
4. Fine-tune for 20 epochs
5. Measure accuracy after fine-tuning (should recover)

In [ ]:
# Exercise starter code
torch.manual_seed(42)

class ExerciseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Generate synthetic data
X = torch.randn(500, 20)
W_true = torch.randn(20, 3)
Y = (X @ W_true).argmax(dim=1)

model = ExerciseNet()

# Your code here:
# Step 1: Train for 100 epochs
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(100):
    loss = nn.functional.cross_entropy(model(X), Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

acc_before = (model(X).argmax(1) == Y).float().mean()
print(f"Before pruning: {acc_before:.2%}")

# Step 2: Prune 50% globally
prune.global_unstructured(
    [(model.fc1, 'weight'), (model.fc2, 'weight'), (model.fc3, 'weight')],
    pruning_method=prune.L1Unstructured,
    amount=0.5,
)
acc_pruned = (model(X).argmax(1) == Y).float().mean()
print(f"After pruning:  {acc_pruned:.2%}")

# Step 3: Fine-tune
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(20):
    loss = nn.functional.cross_entropy(model(X), Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

acc_finetuned = (model(X).argmax(1) == Y).float().mean()
print(f"After finetune: {acc_finetuned:.2%}")

## Key Takeaways

1. **Parametrizations** enforce constraints (symmetric, positive definite, orthogonal) transparently — the constrained weight is computed on-the-fly during forward

2. **Built-in parametrizations** (`spectral_norm`, `orthogonal`) cover common needs; write custom ones with a simple `nn.Module`

3. **Pruning** removes weights by magnitude — **unstructured** (individual weights) vs **structured** (whole neurons/channels)

4. **Global pruning** outperforms per-layer pruning because it allocates capacity where it matters most

5. **Prune → fine-tune** is the standard workflow; pruning without fine-tuning usually degrades accuracy significantly

6. **Sequence packing** (`pack_padded_sequence`) eliminates wasted computation on padding tokens in RNNs

7. **NestedTensors** are PyTorch's native variable-length container — no padding, works with linear layers

8. **Conv-BN fusion** merges two ops into one for inference — mathematically exact, measurably faster

9. **skip_init** avoids redundant weight initialization when loading pre-trained models

10. **parameters_to_vector** flattens all params for model comparison, interpolation, and custom optimizers